# Proyecto · Clase 4 — Diez neuronas: reconocer del 0 al 9

Una neurona responde **una** pregunta de sí/no. Para reconocer 10 dígitos ponemos **10 neuronas**, una por
dígito, que miran la misma imagen. Cada una dice «qué tanto me parece mi número».

Al final de hoy vas a **dibujar tus propios números** y tu red los va a leer.

In [ ]:
import os, urllib.request
for archivo in ["digitos.py", "dibujar.py"]:
    if not os.path.exists(archivo):
        urllib.request.urlretrieve("https://raw.githubusercontent.com/datahuba/curso-redes-neuronales/main/proyecto/" + archivo, archivo)

import numpy as np
import matplotlib.pyplot as plt
from digitos import cargar_mnist, mostrar, ok, guardar_pesos, leer_dibujo

X_train, y_train, X_test, y_test = cargar_mnist()
X, Xt = X_train.reshape(-1, 784) / 255, X_test.reshape(-1, 784) / 255

## 1. La respuesta correcta, en formato «10 casillas» (one-hot)
Con 10 neuronas, la respuesta correcta ya no es un número sino 10 casillas: un 1 en la del dígito y 0 en el resto.

`3` → `[0, 0, 0, 1, 0, 0, 0, 0, 0, 0]`

✏️ **Ejercicio 1.** Pista: `np.eye(10)` es una tabla con un 1 en la diagonal; su fila 3 es justo el one-hot del 3.

In [ ]:
def one_hot(y):
    return ...

Y = one_hot(y_train)
print(y_train[:3]); print(Y[:3])

In [ ]:
ok(one_hot(np.array([3])).tolist() == [[0, 0, 0, 1, 0, 0, 0, 0, 0, 0]], "one_hot(3) tiene un 1 en la casilla 3")

## 2. Softmax: 10 puntajes → 10 probabilidades que suman 1
Cada neurona da un puntaje `z` (cualquier número). **Softmax** los convierte en probabilidades:

$$p_k = \frac{e^{z_k}}{e^{z_0} + e^{z_1} + \dots + e^{z_9}}$$

✏️ **Ejercicio 2.** Para muchas imágenes a la vez (`Z` tiene una fila por imagen). Truco anti-desbordes: resta
primero el máximo de cada fila; el resultado no cambia.

In [ ]:
def softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)
    E = ...
    return ...                       # cada fila dividida entre su suma

print(softmax(np.array([[2.0, 1.0, 0.1]])).round(2))

In [ ]:
P = softmax(np.random.default_rng(0).normal(size=(5, 10)))
ok(np.allclose(P.sum(axis=1), 1) and (P > 0).all(), "cada fila suma 1")

## 3. El modelo y su pendiente
Los pesos ahora son una **tabla** `W` de 784 × 10 (una columna por neurona) y `b` tiene 10 sesgos:

$$P = \text{softmax}(X W + b)$$

Y la pendiente tiene **exactamente la misma forma que en la Clase 3**: error × entrada.

$$\text{grad}_W = \frac{1}{n} X^\top (P - Y) \qquad \text{grad}_b = \text{promedio por columnas de } (P - Y)$$

In [ ]:
def error_promedio(P, Y):
    return -np.mean(np.sum(Y * np.log(P + 1e-9), axis=1))

## 4. Entrenar por lotes (*mini-batches*)
Calcular la pendiente con las 60 000 imágenes para dar **un solo paso** es lento. Mejor: mezclar, tomar de a 128
imágenes, dar un paso con cada grupo. En una época damos ~470 pasitos en vez de 1.

✏️ **Ejercicio 3.** Completa el paso dentro del ciclo.

In [ ]:
def entrenar(X, Y, eta=0.5, epocas=5, lote=128, semilla=0):
    rng = np.random.default_rng(semilla)
    W, b = np.zeros((784, 10)), np.zeros(10)
    historia = []
    for epoca in range(epocas):
        orden = rng.permutation(len(X))
        for i in range(0, len(X), lote):
            Xb, Yb = X[orden[i:i + lote]], Y[orden[i:i + lote]]
            P = ...
            grad_W = ...
            grad_b = ...
            W, b = ..., ...
        historia.append(error_promedio(softmax(X @ W + b), Y))
        print(f"época {epoca + 1}: error {historia[-1]:.3f}")
    return W, b, historia

W, b, historia = entrenar(X, Y)
plt.plot(range(1, len(historia) + 1), historia, "o-"); plt.xlabel("época"); plt.ylabel("error"); plt.show()

pred = softmax(Xt @ W + b).argmax(axis=1)          # la neurona más convencida gana
precision = (pred == y_test).mean()
print(f"Precisión en los 10 dígitos: {precision:.1%}   (las plantillas de la Clase 1 sacaban 82 %)")

In [ ]:
ok(precision > 0.9, "las 10 neuronas aciertan más del 90 %")

## 5. ¿Qué mira cada neurona?
Cada columna de `W` vuelve a ser una imagen. Compáralas con las plantillas de la Clase 1: rojo = «suma» a favor,
azul = «resta».

In [ ]:
fig, ejes = plt.subplots(1, 10, figsize=(14, 2))
for d, ax in enumerate(ejes):
    ax.imshow(W[:, d].reshape(28, 28), cmap="bwr", vmin=-abs(W).max(), vmax=abs(W).max())
    ax.set_title(d); ax.axis("off")
plt.show()

## 6. ¿Con qué se confunde? La matriz de confusión
Fila = número correcto, columna = lo que dijo la red. La diagonal son los aciertos.

In [ ]:
confusion = np.zeros((10, 10), dtype=int)
np.add.at(confusion, (y_test, pred), 1)
plt.imshow(confusion, cmap="Blues")
for i in range(10):
    for j in range(10):
        plt.text(j, i, confusion[i, j], ha="center", va="center", fontsize=8,
                 color="white" if confusion[i, j] > 500 else "black")
plt.xlabel("lo que dijo la red"); plt.ylabel("número correcto"); plt.xticks(range(10)); plt.yticks(range(10))
plt.show()

## 7. ¡A dibujar! ✍️
Guarda los pesos y, **en una terminal dentro de la carpeta `proyecto/`**, ejecuta:
```
python dibujar.py
```
Se abre una ventana: dibuja un número con el mouse y mira qué dice tu red.

**En Colab** no hay ventana: dibuja un número en Paint (fondo negro o blanco, trazo grueso), guárdalo como PNG,
súbelo con el ícono de carpeta de la izquierda y usa la celda de abajo.

In [ ]:
guardar_pesos(W1=W, b1=b)

In [ ]:
if os.path.exists("mi_numero.png"):
    x = leer_dibujo("mi_numero.png")
    p = softmax(x[None] @ W + b)[0]
    mostrar(x, titulos=[f"la red dice: {p.argmax()} ({p.max():.0%})"])

## Para comentar
- ¿Con qué números se equivoca más cuando **tú** dibujas? ¿Y si dibujas el número chiquito o en una esquina?
- Cada neurona solo puede trazar **una línea recta** de separación (por eso a esto se le llama *modelo lineal*).
  Más épocas casi no mejoran: el modelo se queda en ~92 %.

**Clase 5:** ponemos una **capa oculta** entre la imagen y las 10 neuronas, y la precisión salta a ~97 %.